# 균형 데이터셋 생성

**처리 내용**
1. 100개 미만 + valid/test 0개 클래스 제거
2. Phone / Glasses / Sunglasses → train에서 bbox 단위로 언더샘플
3. 클래스 ID 재매핑 후 `dataset_balanced/` 폴더 생성

**제거 클래스**: Charging-cable(6), Earphones(15), Pen(15), Umbrella(117), iPad(9)  
**유지 클래스**: Card · Glasses · Keys · Phone · Sunglasses · Wallet · Watch

In [ ]:
import random, shutil, yaml
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
import pandas as pd

ORIG_DIR     = Path("dataset")
OUT_DIR      = Path("dataset_balanced")
SEED         = 42
random.seed(SEED)

# ── 원본 클래스 정의 ───────────────────────────────────────
ORIG_NAMES = [
    'Card', 'Charging-cable', 'Earphones', 'Glasses', 'Keys',
    'Pen', 'Phone', 'Sunglasses', 'Umbrella', 'Wallet', 'Watch', 'iPad'
]
REMOVE_IDS = {1, 2, 5, 8, 11}   # Charging-cable, Earphones, Pen, Umbrella, iPad

# ── 새 ID 매핑 ─────────────────────────────────────────────
id_map   = {}   # old_id → new_id
new_names = []
new_id = 0
for old_id, name in enumerate(ORIG_NAMES):
    if old_id not in REMOVE_IDS:
        id_map[old_id] = new_id
        new_names.append(name)
        new_id += 1

NC_NEW = len(new_names)

# ── 언더샘플 설정 (train만 적용) ───────────────────────────
# 아래 숫자를 조정해 원하는 목표 bbox 수를 설정하세요
TARGET = 700   # Phone / Glasses / Sunglasses 를 이 수로 줄임

# 언더샘플 대상 (원본 ID 기준)
OVERSAMPLE_ORIG = {3, 6, 7}   # Glasses, Phone, Sunglasses

print("클래스 ID 매핑:")
for old_id, name in enumerate(ORIG_NAMES):
    if old_id in REMOVE_IDS:
        print(f"  [{old_id:2d}] {name:<18} → 제거")
    else:
        tag = f"→ [{id_map[old_id]}] (언더샘플 목표={TARGET})" if old_id in OVERSAMPLE_ORIG else f"→ [{id_map[old_id]}]"
        print(f"  [{old_id:2d}] {name:<18} {tag}")
print(f"\n최종 클래스 ({NC_NEW}개): {new_names}")

In [ ]:
# ── 헬퍼: 레이블 파일 파싱 ────────────────────────────────
def parse_label(path: Path):
    """(class_id, cx, cy, w, h) 튜플 리스트 반환."""
    rows = []
    for line in path.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) == 5:
            rows.append((int(parts[0]), *map(float, parts[1:])))
    return rows


def collect_train_bbox_indices(label_dir: Path):
    """
    train 레이블 전체를 스캔해
    언더샘플 대상 클래스별로 (파일명, 라인 인덱스) 목록 반환.
    """
    class_indices = defaultdict(list)  # orig_id → [(stem, line_idx), ...]
    for lbl_path in sorted(label_dir.glob("*.txt")):
        for i, (cid, *_) in enumerate(parse_label(lbl_path)):
            if cid in OVERSAMPLE_ORIG:
                class_indices[cid].append((lbl_path.stem, i))
    return class_indices


# ── train 언더샘플 keep-set 결정 ──────────────────────────
print("train 레이블 스캔 중...")
class_indices = collect_train_bbox_indices(ORIG_DIR / "train" / "labels")

# 클래스별로 TARGET개 무작위 선택 → keep
keep_set = set()   # (stem, line_idx)
for cid, indices in class_indices.items():
    kept = random.sample(indices, min(TARGET, len(indices)))
    keep_set.update(kept)
    print(f"  {ORIG_NAMES[cid]:<18} {len(indices):>5}개 → {len(kept)}개 유지")

print(f"\n언더샘플 keep-set 크기: {len(keep_set)}")

In [ ]:
# ── 데이터셋 생성 ──────────────────────────────────────────
def process_split(split: str, apply_undersample: bool):
    """
    split 폴더를 처리해 OUT_DIR/{split}/ 에 저장.
    - 제거 클래스 bbox 라인 삭제
    - 언더샘플 클래스 라인 선택적 제거 (train만)
    - 유효 annotation이 하나도 없는 이미지는 건너뜀
    - 클래스 ID 재매핑
    """
    src_img = ORIG_DIR / split / "images"
    src_lbl = ORIG_DIR / split / "labels"
    dst_img = OUT_DIR  / split / "images"
    dst_lbl = OUT_DIR  / split / "labels"
    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)

    saved, skipped = 0, 0
    for lbl_path in sorted(src_lbl.glob("*.txt")):
        rows = parse_label(lbl_path)
        new_lines = []
        for i, (cid, cx, cy, w, h) in enumerate(rows):
            if cid in REMOVE_IDS:
                continue
            if apply_undersample and cid in OVERSAMPLE_ORIG:
                if (lbl_path.stem, i) not in keep_set:
                    continue
            new_cid = id_map[cid]
            new_lines.append(f"{new_cid} {cx} {cy} {w} {h}")

        # annotation이 하나도 없으면 이미지 제외
        if not new_lines:
            skipped += 1
            continue

        # 레이블 저장
        (dst_lbl / lbl_path.name).write_text("\n".join(new_lines))

        # 이미지 복사
        for ext in (".jpg", ".png"):
            img_path = src_img / (lbl_path.stem + ext)
            if img_path.exists():
                shutil.copy(img_path, dst_img / img_path.name)
                break
        saved += 1

    return saved, skipped


if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)

for split in ("train", "valid", "test"):
    saved, skipped = process_split(split, apply_undersample=(split == "train"))
    print(f"[{split}]  저장: {saved}장  |  annotation 없어 제외: {skipped}장")

# data.yaml 생성
balanced_yaml = {
    "train": str(OUT_DIR / "train" / "images"),
    "val"  : str(OUT_DIR / "valid" / "images"),
    "test" : str(OUT_DIR / "test"  / "images"),
    "nc"   : NC_NEW,
    "names": new_names,
}
with open(OUT_DIR / "data.yaml", "w") as f:
    yaml.dump(balanced_yaml, f, allow_unicode=True, default_flow_style=False)

print(f"\ndata.yaml 저장: {OUT_DIR / 'data.yaml'}")

In [ ]:
# ── 결과 검증 ──────────────────────────────────────────────
from collections import Counter

def count_bboxes(split_dir: Path):
    counter = Counter()
    for f in (split_dir / "labels").glob("*.txt"):
        for line in f.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) == 5:
                counter[int(parts[0])] += 1
    return counter

print("=" * 60)
print(f"{'클래스':<18} {'train':>8} {'valid':>8} {'test':>8}")
print("-" * 60)

counts = {
    split: count_bboxes(OUT_DIR / split)
    for split in ("train", "valid", "test")
}
for nid, name in enumerate(new_names):
    tr = counts["train"].get(nid, 0)
    vl = counts["valid"].get(nid, 0)
    te = counts["test"].get(nid, 0)
    print(f"  {name:<16} {tr:>8,} {vl:>8,} {te:>8,}")

print("-" * 60)
for split in ("train", "valid", "test"):
    n_img = len(list((OUT_DIR / split / "images").glob("*")))
    n_bbox = sum(counts[split].values())
    print(f"  {split:<16} {n_bbox:>8,}개  ({n_img:,}장)")
print("=" * 60)

In [ ]:
# ── 전/후 분포 시각화 ──────────────────────────────────────
ORIG_COUNTS = [717, 6, 15, 3882, 144, 15, 5075, 1368, 117, 681, 591, 9]

orig_kept = {
    ORIG_NAMES[old_id]: ORIG_COUNTS[old_id]
    for old_id in range(12)
    if old_id not in REMOVE_IDS
}
new_train = {new_names[nid]: counts["train"].get(nid, 0) for nid in range(NC_NEW)}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 전
axes[0].barh(list(orig_kept.keys()), list(orig_kept.values()),
             color="steelblue", alpha=0.8)
axes[0].set_title("원본 train (유지 클래스만)", fontweight="bold")
axes[0].set_xlabel("bbox 수")
axes[0].invert_yaxis()
for i, v in enumerate(orig_kept.values()):
    axes[0].text(v + 30, i, str(v), va="center", fontsize=8)

# 후
axes[1].barh(list(new_train.keys()), list(new_train.values()),
             color="tomato", alpha=0.8)
axes[1].set_title(f"균형 train (TARGET={TARGET})", fontweight="bold")
axes[1].set_xlabel("bbox 수")
axes[1].invert_yaxis()
for i, v in enumerate(new_train.values()):
    axes[1].text(v + 10, i, str(v), va="center", fontsize=8)

mx_new = max(new_train.values())
mn_new = min(new_train.values())
axes[1].set_title(
    f"균형 train  (불균형 비율 {mx_new/max(mn_new,1):.1f}x)",
    fontweight="bold"
)

plt.suptitle("클래스 분포 전/후 비교 (train bbox)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()